[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 Hard: GPT-2 Transformer Block

Implement a full **GPT-2 style Transformer block** — combining everything you've learned.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.2 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import math

In [3]:
# ✏️ YOUR IMPLEMENTATION HERE

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = self.d_model // self.num_heads

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.mlp = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
        )


    def forward(self, x):
        bs, seq, _ = x.size()

        x_norm1 = self.ln1(x)

        # MHA
        Q = self.W_q(x_norm1).reshape(bs, seq, self.num_heads, self.d_k).transpose(1, 2) # bs, nh, seq, dk
        K = self.W_k(x_norm1).reshape(bs, seq, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x_norm1).reshape(bs, seq, self.num_heads, self.d_k).transpose(1, 2)

        mask = torch.tril(torch.ones(seq, seq).reshape(1, 1, seq, seq))

        scores = Q @ K.mT / math.sqrt(self.d_k) # bs, nh, seq, seq
        scores = scores.masked_fill(mask == 0, float('-inf'))
        x_att = torch.softmax(scores, -1) @ V # bs, nh, seq, dk
        
        x_att = x_att.transpose(1, 2).reshape(bs, seq, -1)
        x_att = self.W_o(x_att)

        x = x + x_att

        x = x + self.mlp(self.ln2(x))

        return x

In [4]:
# 🧪 Debug
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("Output shape:", out.shape)           # (2, 8, 64)
print("Is nn.Module?", isinstance(block, nn.Module))
print("Params:", sum(p.numel() for p in block.parameters()))

Output shape: torch.Size([2, 8, 64])
Is nn.Module? True
Params: 49984


In [5]:
from torch_judge import check
check('gpt2_block')


🧪 Testing: GPT-2 Transformer Block (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (3.6ms)
  ✅ [2/5] Has LayerNorm (pre-norm architecture) (1.3ms)
  ✅ [3/5] MLP has 4x expansion with GELU (0.8ms)
  ✅ [4/5] Causal masking — future doesn't affect past (28.5ms)
  ✅ [5/5] Gradient flow to all parameters (36.3ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (70.5ms total)
  Progress saved. Run status() to see your dashboard.

